## Data Source

The dataset consists of two CSV files exported from Salesforce.  
They were generated as a result of an email campaign with A/B testing (sent vs. not sent).  
Each file contains information about customers and their subscription behavior.

# Imports
This section imports the necessary Python libraries for data processing and analysis.

In [1]:
import pandas as pd
from pathlib import Path

# Data Loading
In this section, two CSV files from the A/B test campaign are loaded and combined into a single dataset.

In [4]:
base_path = Path("../Data/raw")

df1 = pd.read_csv(base_path / "Kampagnenmitglieder_nein_raw.csv", sep=";")
df2 = pd.read_csv(base_path / "Kampagnenmitglieder_ja_raw.csv", sep=";")

# define relevant columns for analysis
relevant_cols = [
    "SAP GP Nummer",
    "letzter Abo - Auftrag: Auftrag von",
    "letzter Abo - Auftrag: Auftrag bis",
    "letzter Abo - Auftrag: Kündigungsgrund Code"
]

# keep only existing columns in each dataset
df1_clean = df1[[col for col in relevant_cols if col in df1.columns]].copy()
df2_clean = df2[[col for col in relevant_cols if col in df2.columns]].copy()

# add treatment indicator (A/B test)
df1_clean["treatment"] = 0  # not sent
df2_clean["treatment"] = 1  # sent

# combine datasets (stack rows)
df = pd.concat([df1_clean, df2_clean], ignore_index=True)

# rename columns for easier handling
df = df.rename(columns={
    "SAP GP Nummer": "customer_id",
    "letzter Abo - Auftrag: Auftrag von": "start_date",
    "letzter Abo - Auftrag: Auftrag bis": "end_date",
    "letzter Abo - Auftrag: Kündigungsgrund Code": "cancel_reason"
})

# preview
df.head()

,customer_id,start_date,end_date,cancel_reason,treatment
0,18640485,05.12.2022,NaN,NaN,0
1,19173245,01.04.2026,NaN,NaN,0
2,19173659,01.01.2021,NaN,NaN,0
3,19173423,01.10.2025,NaN,NaN,0
4,19173333,28.10.2024,NaN,NaN,0


# Data Cleaning

In [11]:
# convert date columns to datetime
df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
df["end_date"] = pd.to_datetime(df["end_date"], errors="coerce")



In [20]:
# define observation window for post-campaign churn
start = pd.Timestamp("2026-03-19")
end = pd.Timestamp("2026-04-19")

# create churn variable: 1 if subscription ended within the observation window
df["churn"] = (
    df["end_date"].notna() &
    (df["end_date"] >= start) &
    (df["end_date"] <= end)
).astype(int)

# overall churn rate (%)
overall_churn = df["churn"].mean() * 100

# overall churn by treatment group (%)
churn_by_group = df.groupby("treatment")["churn"].mean() * 100
churn_by_group.index = ["Not sent", "Sent"]

# KA churn based on the same base: all customers in each group
ka_churn_by_group = df.groupby("treatment").apply(
    lambda x: ((x["churn"] == 1) & (x["cancel_reason"] == "KA")).mean() * 100
)
ka_churn_by_group.index = ["Not sent", "Sent"]

# share of KA among all churned customers (%)
ka_share_among_churned = (
    (df.loc[df["churn"] == 1, "cancel_reason"] == "KA").mean() * 100
)

# print results
print(f"Overall churn: {overall_churn:.2f}%\n")

print("Overall churn by treatment group (%):")
print(churn_by_group.round(2))

print("\nKA churn by treatment group (% of all customers):")
print(ka_churn_by_group.round(2))

print(f"\nShare of KA among all churned customers: {ka_share_among_churned:.2f}%")

Overall churn: 1.04%

Overall churn by treatment group (%):
Not sent    1.48
Sent        0.94
Name: churn, dtype: float64

KA churn by treatment group (% of all customers):
Not sent    0.17
Sent        0.25
dtype: float64

Share of KA among all churned customers: 22.90%


In [28]:
# filter only customers who churned in the campaign window
df_window = df[
    (df["churn"] == 1)
].copy()

# count per cancel_reason (only in this month)
counts = df_window["cancel_reason"].value_counts()

# one example customer per reason
examples = (
    df_window[["cancel_reason", "customer_id"]]
    .dropna(subset=["cancel_reason"])
    .drop_duplicates(subset=["cancel_reason"])
    .set_index("cancel_reason")
)

# combine
df_reason_map = (
    counts.to_frame("count")
    .join(examples)
    .reset_index()
    .rename(columns={"index": "cancel_reason"})
)
df_window.groupby(["treatment", "cancel_reason"]).size().reset_index(name="count")

df_reason_map


,cancel_reason,count,customer_id
0,BEF,69,19250158
1,KA,68,19272406
2,PRE,49,19316345
3,TOD,16,19269995
4,KZE,16,19568358
5,KZA,15,19614244
6,AME,14,19248088
7,AKP,10,16102967
8,INH,9,19215867
9,KIN,9,19541568


In [30]:
# define observation window
start = pd.Timestamp("2026-03-19")
end = pd.Timestamp("2026-04-18")

# filter only churned customers in this window
df_window = df[
    (df["end_date"].notna()) &
    (df["end_date"] >= start) &
    (df["end_date"] <= end)
].copy()

# group by treatment and cancel_reason (counts)
result = (
    df_window
    .groupby(["treatment", "cancel_reason"])
    .size()
    .reset_index(name="count")
)

pivot_pct = (
    df_window
    .groupby(["treatment", "cancel_reason"])
    .size()
    .groupby(level=0)
    .apply(lambda x: x / x.sum() * 100)
    .unstack(fill_value=0)
    .T
)

pivot_pct.rename(columns={0: "Not sent", 1: "Sent"}).round(2)

treatment,Not sent,Sent
treatment,Not sent,Sent
cancel_reason,,
AKP,6.58,2.42
AME,3.95,4.35
BEF,23.68,24.15
BON,1.32,0.48
FIN,0.00,0.97
GPW,1.32,0.00
INH,5.26,2.42
KA,11.84,26.57
